In [62]:
# Preparing pathing
%load_ext autoreload
%autoreload 2
from titanic_ml import paths
import matplotlib.pyplot as plt
import pandas as pd
from titanic_ml.common.data.eda import summarize_categorical_column, summarize_numerical_column
from titanic_ml.common.data.eda import run_eda 
TARGET = "Survived"

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


Experiment =[
    Name,
    Model_name,
    Model_params:{},
    feature_engineering:{},
    features:{numerical:{}, onehot:{}, ordinal:{}},
    preprocessing:{},
    evaluation:{},
    notes,
]

### Run_Experiments Blueprint - WIP:

run_experiments (Df, Experiments) -> Result_df for each experiment:

    Feature engineering function (DF, Experiments[feature_engineering]) -> this experiments modified_df:
        (creates/modifies columns)

    build_preprocessor(Experiment[[features, preprocessing,]]) -> preprocessor:
        (prepares selected columns for sklearn model)

    model(modified_df, preprocessor, Experiment[Model_name, Model_params, evaluation]) -> Prediction and evaluation:
        (trains/predicts/evaluates)

In [63]:
def Titanic_feature_engineering(df):

    df = df.copy()

    # Creating 'Deck' and 'Has_Cabin'
    df['Deck'] = df['Cabin'].str[0]
    df['Has_Cabin'] = df['Deck'].notnull().astype(int)

    # Creating 'Title'
    title_name = df['Name'].str.split(',').str[1]
    title = title_name.str.split('.').str[0]
    df['Title'] = title.str.strip()

    # Dropping 'PassengerId', 'Ticket', 'Cabin', 'Name'
    df.drop(['PassengerId', 'Ticket', 'Cabin', 'Name'], axis=1, inplace=True)

    # Creating 'Family_size', 'Alone', 'IsAgeMissing'
    df['Family_size'] = df['SibSp'] + df['Parch'] + 1
    df['Alone'] = (df['Family_size'] == 1).astype(int)
    df['IsAgeMissing'] = df['Age'].isna().astype(int)

    # Cutting Age to create Age_bin
    bins = [0, 14, 35, 60, 100]
    labels = ['0', '2', '3', '1']
    df['Age_bin'] = pd.cut(df['Age'], bins=bins, labels=labels, right=False)

    # Creating 'Fare/Family_size', 'Fare/Age', 'Fare_bin'
    df['Fare/Family_size'] = df['Fare'] / df['Family_size']
    df['Fare/Age'] = df['Fare'] / df['Age']
    df['Fare_bin'] = pd.qcut(df['Fare'], q = 5, labels=['0','1', '2', '3', '4'])

    # Creating 'Pclass_Sex', 'Pclass_Title'
    df['Pclass_Sex'] = df['Pclass'].astype(str) + '_' + df['Sex'].astype(str)
    df['Pclass_Title'] = df['Pclass'].astype(str) + '_' + df['Title']

    # Filling Deck's Nan with 'None'
    df['Deck'] = df['Deck'].fillna('None')

    

    return df

In [64]:
from titanic_ml.common.experiments.runner import run_experiments
from titanic_ml.common.experiments import experiment_config
from titanic_ml.common.experiments.experiment_report import experiment_report

df = pd.read_csv(paths.TRAIN_PATH)
# print(df.head())

# working_df = Titanic_feature_engineering(df)
# print(working_df.head())

config = experiment_config.dummy
# print(baseline)




In [65]:
result = run_experiments(df, config, target=TARGET)
individual_report, full_report = experiment_report(result, config, print_report=True)

Individual Experiment Reports:

Experiment - baseline_logreg:
| Field | Value |
|---|---|
|Train accuracy| 0.803 ± 0.005 |
|Train precision| 0.762 ± 0.011 |
|Train recall| 0.708 ± 0.015 |
|Train f1| 0.734 ± 0.008 |
|Test accuracy| 0.786 ± 0.018 |
|Test precision| 0.736 ± 0.036 |
|Test recall| 0.693 ± 0.038 |
|Test f1| 0.713 ± 0.026 |

Full configuration:
```python
{'name': 'baseline_logreg',
 'features': ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked'],
 'feature_engineering': None,
 'preprocessing': {'numeric_features': ['Age', 'SibSp', 'Parch', 'Fare'],
                   'onehot_features': ['Sex', 'Embarked'],
                   'ordinal_features': ['Pclass'],
                   'numeric_imputer': 'median',
                   'categorical_imputer': 'most_frequent',
                   'scaler': 'standard'},
 'model_name': 'logreg',
 'model_params': {'max_iter': 1000, 'random_state': 42},
 'evaluation': {'method': 'cross_validation',
                'cv': 5,
            